In [ ]:
#Imports to speed up the dataprocessing with Rapids, if no cuda is available it falls back to CPU

import shutil
import subprocess
import yaml
from pathlib import Path

import numpy as np
import pandas as cpu_pd

pd = cpu_pd
cp = None
cudf = None
USE_RAPIDS = False
DATA_BACKEND = "CPU pandas"
RAPIDS_UNAVAILABLE_REASON = ""


def _nvidia_gpu_available():
    if shutil.which("nvidia-smi") is None:
        return False, "nvidia-smi was not found"

    try:
        result = subprocess.run(
            ["nvidia-smi", "-L"],
            capture_output=True,
            text=True,
            timeout=5,
        )
    except (OSError, subprocess.SubprocessError) as exc:
        return False, f"nvidia-smi check failed: {exc}"

    if result.returncode != 0:
        message = result.stderr.strip() or result.stdout.strip() or "nvidia-smi returned an error"
        return False, message

    if "GPU" not in result.stdout:
        return False, "nvidia-smi did not report an NVIDIA GPU"

    return True, result.stdout.strip().splitlines()[0]


def _enable_rapids_if_available():
    global cudf, cp, pd, USE_RAPIDS, DATA_BACKEND, RAPIDS_UNAVAILABLE_REASON

    has_gpu, gpu_reason = _nvidia_gpu_available()
    if not has_gpu:
        RAPIDS_UNAVAILABLE_REASON = gpu_reason
        return

    try:
        import cudf as gpu_cudf
        import cupy as gpu_cp

        if gpu_cp.cuda.runtime.getDeviceCount() < 1:
            RAPIDS_UNAVAILABLE_REASON = "CuPy did not find a usable CUDA device"
            return
    except Exception as exc:
        RAPIDS_UNAVAILABLE_REASON = f"RAPIDS/cuDF is not usable in this environment: {exc}"
        return

    cudf = gpu_cudf
    cp = gpu_cp
    pd = cudf
    USE_RAPIDS = True
    DATA_BACKEND = "GPU RAPIDS/cuDF"
    RAPIDS_UNAVAILABLE_REASON = gpu_reason


def switch_to_cpu(reason, *frames):
    global cp, pd, USE_RAPIDS, DATA_BACKEND

    if USE_RAPIDS:
        print(f"Switching to CPU pandas: {reason}")

    pd = cpu_pd
    cp = None
    USE_RAPIDS = False
    DATA_BACKEND = "CPU pandas"

    converted = []
    for frame in frames:
        converted.append(frame.to_pandas() if hasattr(frame, "to_pandas") else frame)

    if len(converted) == 1:
        return converted[0]

    return tuple(converted)


_enable_rapids_if_available()
print(f"Using {DATA_BACKEND} for data handling")
if not USE_RAPIDS:
    print(f"RAPIDS fallback reason: {RAPIDS_UNAVAILABLE_REASON}")

In [ ]:
# Importing the config file to use values

config_candidates = [
    Path("config.yaml"),
    Path("../config.yaml"),
    Path("configs/data_config.yaml"),
    Path("../configs/data_config.yaml"),
]

config_path = next((path for path in config_candidates if path.exists()), None)
if config_path is None:
    raise FileNotFoundError("Could not find config.yaml or configs/data_config.yaml")

project_root = config_path.parent.parent if config_path.parent.name == "configs" else Path(".")


def project_path(path_value):
    path = Path(path_value)
    if path.is_absolute():
        return path
    return project_root / path

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

config

In [ ]:
#Since the dataset contains ? and are seperated by ; this needs to be handled to proceed

read_csv_kwargs = {"sep": ";", "na_values": "?"}
if not USE_RAPIDS:
    read_csv_kwargs["low_memory"] = False

input_path = project_path(config["data"]["input_path"])

try:
    df = pd.read_csv(input_path, **read_csv_kwargs)
except Exception as exc:
    if USE_RAPIDS:
        switch_to_cpu(f"RAPIDS read_csv failed ({exc})")
        df = pd.read_csv(input_path, sep=";", na_values="?", low_memory=False)
    else:
        raise

In [ ]:
df.info()

In [ ]:
method = config["data"].get("fill_missing")
method_key = None if method is None else str(method).lower()


def apply_missing_value_strategy(frame, strategy):
    if strategy == "drop":
        return frame.dropna()

    if strategy == "ffill":
        return frame.ffill()

    if strategy == "bfill":
        return frame.bfill()

    if strategy in (None, "none", ""):
        return frame

    raise ValueError("Invalid fill_missing option in config")


try:
    df = apply_missing_value_strategy(df, method_key)
except Exception as exc:
    if USE_RAPIDS:
        df = switch_to_cpu(f"RAPIDS missing-value handling failed ({exc})", df)
        df = apply_missing_value_strategy(df, method_key)
    else:
        raise

In [ ]:
#Its assumed that withNaN means the application is not drawing power or the Sub-meter didnt record anything so the datapoint is removed. Since this is only 1% and will be aggragted into hours its not seen as a problem to remove them

try:
    df = df.dropna()
except Exception as exc:
    if USE_RAPIDS:
        df = switch_to_cpu(f"RAPIDS dropna failed ({exc})", df)
        df = df.dropna()
    else:
        raise

In [ ]:
#To predict the hourly total household energy consumption, the data is converted into the correct time format and then aggreated each x amount of time, Some of the features represent instantaneous measurements so they are averaged instead.

cols = [
    "Global_active_power", "Global_reactive_power", "Voltage", "Global_intensity",
    "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"
]

resample_rule = config["data"]["time_interval"]

agg_dict = {
    "Global_active_power": "mean",
    "Global_reactive_power": "mean",
    "Voltage": "mean",
    "Global_intensity": "mean",
    "Sub_metering_1": "sum",
    "Sub_metering_2": "sum",
    "Sub_metering_3": "sum"
}


def build_resampled_dataframe(frame):
    frame = frame.copy()
    frame["datetime"] = pd.to_datetime(
        frame["Date"] + " " + frame["Time"],
        format="%d/%m/%Y %H:%M:%S",
        errors="coerce",
    )

    frame = frame.dropna(subset=["datetime"]).set_index("datetime")

    for column in cols:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    return frame.resample(resample_rule).agg(agg_dict)


try:
    hourly_df = build_resampled_dataframe(df)
except Exception as exc:
    if USE_RAPIDS:
        df = switch_to_cpu(f"RAPIDS datetime conversion or resampling failed ({exc})", df)
        hourly_df = build_resampled_dataframe(df)
    else:
        raise

In [ ]:
#Since time is cyclical the data needs to be converted using cos/sin 

def add_cyclical_time_features(frame):
    frame["hour"] = frame.index.hour
    frame["day_of_week"] = frame.index.dayofweek
    frame["month"] = frame.index.month

    if USE_RAPIDS:
        def gpu_cycle(series, period):
            values = series.astype("float64").to_cupy()
            angle = 2 * np.pi * values / period
            return cp.sin(angle), cp.cos(angle)

        frame["hour_sin"], frame["hour_cos"] = gpu_cycle(frame["hour"], 24)
        frame["dow_sin"], frame["dow_cos"] = gpu_cycle(frame["day_of_week"], 7)
        frame["month_sin"], frame["month_cos"] = gpu_cycle(frame["month"], 12)
    else:
        frame["hour_sin"] = np.sin(2 * np.pi * frame["hour"] / 24)
        frame["hour_cos"] = np.cos(2 * np.pi * frame["hour"] / 24)
        frame["dow_sin"] = np.sin(2 * np.pi * frame["day_of_week"] / 7)
        frame["dow_cos"] = np.cos(2 * np.pi * frame["day_of_week"] / 7)
        frame["month_sin"] = np.sin(2 * np.pi * frame["month"] / 12)
        frame["month_cos"] = np.cos(2 * np.pi * frame["month"] / 12)

    return frame


try:
    hourly_df = add_cyclical_time_features(hourly_df)
except Exception as exc:
    if USE_RAPIDS:
        hourly_df = switch_to_cpu(f"RAPIDS cyclical feature creation failed ({exc})", hourly_df)
        hourly_df = add_cyclical_time_features(hourly_df)
    else:
        raise

In [ ]:
# Build the one-step-ahead target used by the GRU and drop the last row which becomes NaN because there is no target
target_col = "target_next_hour"
feature_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
    "hour",
    "day_of_week",
    "month",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
]

hourly_df[target_col] = hourly_df["Global_active_power"].shift(-1)
hourly_df = hourly_df.dropna(subset=feature_cols + [target_col])

X = hourly_df[feature_cols]
y = hourly_df[[target_col]]
gru_input_size = len(feature_cols)

processed_output_path = project_path(
    config["data"].get("processed_output_path", "data/processed/household_power_gru.csv")
)
processed_output_path.parent.mkdir(parents=True, exist_ok=True)

training_df = hourly_df[feature_cols + [target_col]]
if hasattr(training_df, "to_pandas"):
    training_df = training_df.to_pandas()

training_df.to_csv(processed_output_path, index=True, index_label="datetime")

print(f"GRU input_size: {gru_input_size}")
print(f"Target column: {target_col}")
print(f"Saved processed training data to: {processed_output_path}")